# Part 5 — Cumulative Exposure Rescue Model

## Purpose

Parts 1–4 used an **instantaneous threshold** rule: a nucleus was classified as rescued when its signal concentration exceeded a fixed value at a particular timestep.

Part 5 introduces a more biologically plausible alternative:

> Rescue may depend on sustained molecular exposure over time, not only on a single high signal peak.

This keeps the graph-Laplacian diffusion model from Parts 2–4, but adds a second state variable: cumulative exposure.

## Mathematical model

The signal dynamics remain:

$$\nu_{t+1} = \nu_t + \Delta t\left(-\alpha L\nu_t - \beta \nu_t + q\right)$$

where:

- $\nu_t$ is the signal concentration vector at timestep $t$
- $L = D - A$ is the graph Laplacian
- $A$ is the adjacency matrix
- $D$ is the degree matrix
- $\alpha$ is the diffusion strength
- $\beta$ is the decay rate
- $q$ is the source-production vector
- $\Delta t$ is the timestep size

The cumulative exposure state is:

$$e_{t+1} = e_t + \nu_{t+1}\Delta t$$

In this notebook the default convention is **post-update exposure**, meaning exposure is accumulated from the signal after the Euler update. This matches the original Part 5 implementation. A `pre_update` option is also provided for numerical sensitivity checks.

The rescue rule is:

$$r_i(t) = \mathbb{1}\left[e_i(t) \geq \tau_E\right]$$

where $\tau_E$ is the cumulative exposure threshold.

## Step 1 — Imports

The modelling functions now live in `src/`. The notebook remains the experimental and explanatory layer.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.part3_geometry import generate_positions
from src.part5_cumulative import (
    compare_instantaneous_and_cumulative,
    run_cumulative_sweep,
    simulate_cumulative_exposure,
)

## Step 2 — Generate nuclear geometry

The fibre is represented as a one-dimensional spatial domain. Nuclei are placed along this domain with small irregular perturbations, then connected if they fall within a local interaction radius.

In [ ]:
positions = generate_positions(n_nodes=30, length=1.0, jitter=0.01, seed=7)
positions[:5], positions[-5:]

## Step 3 — Run the cumulative exposure model

The default source pattern is a left-side cluster of corrected nuclei. This is intentionally deterministic so that parameter changes can be interpreted cleanly.

In [ ]:
result = simulate_cumulative_exposure(
    positions=positions,
    corrected_fraction=0.1,
    correction_mode="left_cluster",
    radius=0.09,
    alpha=0.12,
    beta=0.03,
    production_rate=0.05,
    dt=0.1,
    steps=300,
    exposure_threshold=1.5,
    exposure_timing="post_update",
)

summary = {
    "rescued_fraction": result["rescued_fraction"],
    "number_rescued": int(result["rescued"].sum()),
    "max_signal": float(result["signal"].max()),
    "mean_signal": float(result["signal"].mean()),
    "max_exposure": float(result["exposure"].max()),
    "mean_exposure": float(result["exposure"].mean()),
}
summary

## Step 4 — Plot final signal and cumulative exposure

A nucleus can have a modest final signal but still accumulate enough exposure to cross the rescue threshold. This is the key conceptual difference from the instantaneous model.

In [ ]:
def plot_signal_and_exposure(result):
    positions = result["positions"]
    signal = result["signal"]
    exposure = result["exposure"]
    rescued = result["rescued"]
    corrected = result["corrected"]
    threshold = result["parameters"]["exposure_threshold"]

    fig, ax1 = plt.subplots(figsize=(10, 4))

    ax1.plot(positions, signal, marker="o", label="Final signal")
    ax1.scatter(positions[corrected], signal[corrected], marker="s", s=90, label="Corrected nuclei")
    ax1.set_xlabel("Position along fibre")
    ax1.set_ylabel("Final signal")
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(positions, exposure, marker="s", linestyle="--", label="Cumulative exposure")
    ax2.axhline(threshold, linestyle=":", label="Exposure threshold")
    ax2.scatter(positions[rescued], exposure[rescued], marker="x", s=90, label="Rescued nuclei")
    ax2.set_ylabel("Cumulative exposure")

    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

    plt.title("Part 5: final signal and cumulative exposure")
    plt.show()

plot_signal_and_exposure(result)

## Step 5 — Exposure accumulation heatmap

The exposure state is cumulative, so it can keep increasing even after the signal profile has largely stabilised.

In [ ]:
def plot_exposure_heatmap(result):
    exposure_history = result["exposure_history"]

    plt.figure(figsize=(10, 5))
    plt.imshow(exposure_history, aspect="auto", origin="lower")
    plt.colorbar(label="Cumulative exposure")
    plt.xlabel("Nucleus index")
    plt.ylabel("Timestep")
    plt.title("Part 5: cumulative exposure over time")
    plt.show()

plot_exposure_heatmap(result)

## Step 6 — Rescue outcome over physical position

This plot separates the biological interpretation from raw signal intensity: rescue is now a memory-like property of accumulated exposure.

In [ ]:
def plot_rescue_by_position(result):
    positions = result["positions"]
    rescued = result["rescued"]
    corrected = result["corrected"]

    y = rescued.astype(int)

    plt.figure(figsize=(10, 2.8))
    plt.scatter(positions, y, s=90, label="Nuclei")
    plt.scatter(positions[corrected], y[corrected], marker="s", s=120, label="Corrected nuclei")
    plt.yticks([0, 1], ["Not rescued", "Rescued"])
    plt.xlabel("Position along fibre")
    plt.title("Part 5: rescue outcome using cumulative exposure")
    plt.grid(True, axis="x", alpha=0.3)
    plt.legend()
    plt.show()

plot_rescue_by_position(result)

## Step 7 — Compare instantaneous rescue and cumulative exposure rescue

The Part 4 instantaneous model uses:

$$r_i(t) = \mathbb{1}\left[\nu_i(t) \geq \tau_S\right]$$

The Part 5 cumulative model uses:

$$r_i(t) = \mathbb{1}\left[e_i(t) \geq \tau_E\right]$$

This comparison is useful because it shows whether rescue is driven by peak signal alone or by sustained low-level exposure.

In [ ]:
comparison = compare_instantaneous_and_cumulative(
    positions=positions,
    corrected_fraction=0.1,
    correction_mode="left_cluster",
    radius=0.09,
    alpha=0.12,
    beta=0.03,
    production_rate=0.05,
    dt=0.1,
    steps=300,
    rescue_threshold=0.5,
    exposure_threshold=1.5,
)

comparison_summary = {
    "instantaneous_rescued_fraction": comparison["instantaneous"]["rescued_fraction"],
    "cumulative_rescued_fraction": comparison["cumulative"]["rescued_fraction"],
    "instantaneous_number_rescued": int(comparison["instantaneous"]["rescued"].sum()),
    "cumulative_number_rescued": int(comparison["cumulative"]["rescued"].sum()),
}
comparison_summary

In [ ]:
def plot_model_comparison(comparison):
    positions = comparison["cumulative"]["positions"]
    inst = comparison["instantaneous"]["rescued"].astype(int)
    cum = comparison["cumulative"]["rescued"].astype(int)

    plt.figure(figsize=(10, 3.2))
    plt.scatter(positions, inst, marker="o", s=90, label="Instantaneous rescue")
    plt.scatter(positions, cum + 0.08, marker="x", s=90, label="Cumulative exposure rescue")
    plt.yticks([0, 1], ["Not rescued", "Rescued"])
    plt.xlabel("Position along fibre")
    plt.title("Part 4 versus Part 5 rescue classification")
    plt.grid(True, axis="x", alpha=0.3)
    plt.legend()
    plt.show()

plot_model_comparison(comparison)

## Step 8 — Sensitivity sweep

Part 5 should not only produce one result. A better modelling question is:

> How stable is the rescue outcome under changes in diffusion, decay, source fraction, and exposure threshold?

The sweep below varies four biologically meaningful parameters:

- $\alpha$: diffusion strength
- $\beta$: decay rate
- $f_c$: corrected-nucleus fraction
- $\tau_E$: exposure threshold

In [ ]:
parameter_grid = {
    "alpha": [0.06, 0.12, 0.24],
    "beta": [0.01, 0.03, 0.06],
    "corrected_fraction": [0.1, 0.2],
    "exposure_threshold": [1.0, 1.5, 2.0],
}

sweep_results = run_cumulative_sweep(
    positions,
    parameter_grid=parameter_grid,
    base_parameters={
        "correction_mode": "left_cluster",
        "radius": 0.09,
        "production_rate": 0.05,
        "dt": 0.1,
        "steps": 300,
    },
)

sweep_results[:5]

In [ ]:
def plot_sweep_alpha_threshold(sweep_results):
    filtered = [row for row in sweep_results if row["beta"] == 0.03 and row["corrected_fraction"] == 0.1]
    thresholds = sorted({row["exposure_threshold"] for row in filtered})

    plt.figure(figsize=(8, 4))
    for threshold in thresholds:
        rows = sorted([row for row in filtered if row["exposure_threshold"] == threshold], key=lambda x: x["alpha"])
        plt.plot(
            [row["alpha"] for row in rows],
            [row["rescued_fraction"] for row in rows],
            marker="o",
            label=f"threshold={threshold}",
        )

    plt.xlabel("Diffusion strength alpha")
    plt.ylabel("Rescued fraction")
    plt.title("Cumulative rescue sensitivity to diffusion and exposure threshold")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

plot_sweep_alpha_threshold(sweep_results)

## Step 9 — Numerical convention check

There are two defensible exposure conventions:

$$e_{t+1} = e_t + \nu_t\Delta t$$

or:

$$e_{t+1} = e_t + \nu_{t+1}\Delta t$$

The second version is used by default because it matches the original notebook. The first version is useful as a sensitivity check. In practice, the difference should become small when $\Delta t$ is sufficiently small.

In [ ]:
pre_update = simulate_cumulative_exposure(
    positions,
    exposure_timing="pre_update",
    exposure_threshold=1.5,
    steps=300,
)

post_update = simulate_cumulative_exposure(
    positions,
    exposure_timing="post_update",
    exposure_threshold=1.5,
    steps=300,
)

{
    "pre_update_rescued_fraction": pre_update["rescued_fraction"],
    "post_update_rescued_fraction": post_update["rescued_fraction"],
    "mean_exposure_difference": float(np.mean(np.abs(pre_update["exposure"] - post_update["exposure"]))),
}

## Interpretation

Part 5 changes the biological meaning of rescue.

The instantaneous model asks whether a nucleus receives enough signal at one point in time. The cumulative model asks whether the nucleus receives enough total exposure across time.

Main implications:

- Weak but persistent signal can now matter.
- Rescue is no longer tied only to the final signal profile.
- Spatial proximity still matters because exposure depends on graph diffusion.
- The model remains deterministic, so parameter effects are interpretable.
- The exposure threshold $\tau_E$ is phenomenological. It is useful for modelling, but it is not yet experimentally fitted.

The next natural extension is an animated simulation showing signal diffusion, exposure accumulation, and rescue classification over time.